<a href="https://colab.research.google.com/github/epi24/multimodal-meme-analysis/blob/main/image_only_resnet_NEW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torchvision import models, transforms
from tqdm.auto import tqdm
from PIL import Image
import gc




PATH_TRAIN_JSON = '/content/drive/MyDrive/meme_train.json'
PATH_VAL_JSON   = '/content/drive/MyDrive/meme_val.json'
PATH_IMAGES     = '/content/drive/MyDrive/all_memes/kym_memes'
SAVE_DIR        = '/content/drive/MyDrive/image_only_resnet_NEW'
MODEL_NAME      = 'resnet50'

CHECKPOINT_PATH = '/content/drive/MyDrive/image_only_resnet_NEW/resnet50_epoch_11.pth'

# HYPERPARAMETER
BATCH_SIZE    = 4500
EPOCHS        = 20
LEARNING_RATE = 1e-4    # ResNet verträgt oft eine etwas höhere Lernrate als CLIP
NUM_WORKERS = 12
PREFETCH_FACTOR = 2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ==========================================
# 2. DATASET MIT RESNET-TRANSFORMS
# ==========================================
class ResNetDataset(Dataset):
    def __init__(self, json_path, img_base_path, is_train=True):
        self.img_base_path = img_base_path
        self.samples = []
        self.label_map = {}
        self.id_to_label = {}

        # --- RESNET STANDARD TRANSFORMATIONEN ---
        # ResNet erwartet exakt 224x224 Pixel und eine bestimmte Normalisierung
        if is_train:
            self.transform = transforms.Compose([
                transforms.Resize((256, 256)),
                transforms.RandomCrop(224), # Datenausweitung
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomRotation(degrees=10),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])
        else:
            self.transform = transforms.Compose([
                transforms.Resize((224, 224)), # Nur skalieren, kein RandomCrop
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])

        print(f"--- [DATASET] Lade {json_path.split('/')[-1]}... ---")
        with open(json_path, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)

        unique_labels = sorted(list(set(item['label'] for item in raw_data)))
        for idx, label in enumerate(unique_labels):
            self.label_map[label] = idx
            self.id_to_label[idx] = label

        # Create SAVE_DIR if it doesn't exist before writing the map file
        os.makedirs(SAVE_DIR, exist_ok=True)
        with open(os.path.join(SAVE_DIR, f"{MODEL_NAME}_map.json"), 'w') as f:
            json.dump(self.id_to_label, f)

        for item in tqdm(raw_data, desc="Lade Bildpfade"):
            label_str = item.get('label')
            filename = item.get('filename')
            full_img_path = os.path.join(self.img_base_path, label_str, filename)

            if not os.path.exists(full_img_path): continue

            self.samples.append({
                'img_path': full_img_path,
                'label': self.label_map[label_str]
            })

        print(f"-> Bereit: {len(self.samples)} Bilder geladen.\n")
        del raw_data
        gc.collect()

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        try:
            image = Image.open(sample['img_path']).convert("RGB")
            pixel_values = self.transform(image)
        except:
            return self.__getitem__((idx + 1) % len(self.samples))

        return {
            'pixel_values': pixel_values,
            'label': torch.tensor(sample['label'], dtype=torch.long)
        }

# ==========================================
# 3. ARCHITEKTUR (RESNET-50)
# ==========================================
class ResNetMemeClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        # Lade ein vortrainiertes ResNet50
        self.resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

        # --- PARTIAL UNFREEZING ---
        # Wir frieren die frühen Schichten ein (die erkennen nur Kanten/Farben)
        for param in self.resnet.parameters():
            param.requires_grad = False

        # Wir tauen den letzten großen Block (layer4) auf, damit das Modell "Memes" lernen kann
        for param in self.resnet.layer4.parameters():
            param.requires_grad = True

        # Wir passen die letzte Schicht (Classifier) an unsere Meme-Klassen an
        num_features = self.resnet.fc.in_features # Bei ResNet50 sind das 2048

        self.resnet.fc = nn.Sequential(
            nn.Dropout(0.5), # Gegen Overfitting
            nn.Linear(num_features, num_classes)
        )

    def forward(self, pixel_values):
        return self.resnet(pixel_values)

# ==========================================
# 4. TRAINING LOOP
# ==========================================
def run_resnet_training():
    print("--- Start ResNet-50 Image-Only Training ---")

    train_dataset = ResNetDataset(PATH_TRAIN_JSON, PATH_IMAGES, is_train=True)
    val_dataset   = ResNetDataset(PATH_VAL_JSON, PATH_IMAGES, is_train=False)

    num_classes = len(train_dataset.label_map)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=PREFETCH_FACTOR)
    val_loader   = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=False,
        prefetch_factor=PREFETCH_FACTOR)

    model = ResNetMemeClassifier(num_classes).to(DEVICE)

    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler('cuda')

    start_epoch = 0

    if CHECKPOINT_PATH and os.path.exists(CHECKPOINT_PATH):
        print(f"\n[INFO] Lade Checkpoint: {CHECKPOINT_PATH}")
        checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)

        if "model_state_dict" in checkpoint:
            model.load_state_dict(checkpoint["model_state_dict"])
            optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
            start_epoch = checkpoint["epoch"]
            best_acc = checkpoint.get("best_acc", 0.0)
            print(f"[SUCCESS] Geladen! Starte ab Epoche {start_epoch+1}.")
        else:
            model.load_state_dict(checkpoint)
            print("[WARNUNG] Alter Checkpoint (Nur Gewichte).")

    print(f"\nStarte Training bis Epoche {EPOCHS}...\n")

    for epoch in range(start_epoch, EPOCHS):
        model.train()
        train_loss = 0
        pbar = tqdm(train_loader, desc=f"Epoche {epoch+1}/{EPOCHS} [Train]")

        for batch in pbar:
            optimizer.zero_grad()

            pixel_values = batch['pixel_values'].to(DEVICE)
            labels = batch['label'].to(DEVICE)

            with torch.amp.autocast('cuda'):
                logits = model(pixel_values)
                loss = criterion(logits, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})

        del pixel_values, labels, logits
        gc.collect()
        torch.cuda.empty_cache()

        model.eval()
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoche {epoch+1} [Valid]", leave=False):
                pixel_values = batch['pixel_values'].to(DEVICE)
                labels = batch['label'].to(DEVICE)

                with torch.amp.autocast('cuda'):
                    logits = model(pixel_values)

                _, preds = torch.max(logits, 1)
                val_total += labels.size(0)
                val_correct += (preds == labels).sum().item()

                del pixel_values, labels, logits

        val_acc = val_correct / val_total
        print(f" -> Resultat E{epoch+1}: Val Acc: {val_acc:.2%}")

        checkpoint_dict = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        }

        torch.save(checkpoint_dict, os.path.join(SAVE_DIR, f"{MODEL_NAME}_epoch_{epoch+1}.pth"))


if __name__ == "__main__":
    run_resnet_training()

--- Start ResNet-50 Image-Only Training ---
--- [DATASET] Lade meme_train.json... ---


Lade Bildpfade:   0%|          | 0/75765 [00:00<?, ?it/s]

-> Bereit: 75763 Bilder geladen.

--- [DATASET] Lade meme_val.json... ---


Lade Bildpfade:   0%|          | 0/9402 [00:00<?, ?it/s]

-> Bereit: 9402 Bilder geladen.

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


...

Epoche 20/20 [Train]:   0%|          | 0/17 [00:00<?, ?it/s]

Exception in thread Exception in thread QueueFeederThread:
Exception in thread QueueFeederThread:
Exception ignored in: <function _ConnectionBase.__del__ at 0x7925499080e0>
Traceback (most recent call last):
  File "/usr/lib/python3.13/multiprocessing/connection.py", line 138, in __del__
Exception in thread QueueFeederThread:
Traceback (most recent call last):
  File "/usr/lib/python3.13/multiprocessing/queues.py", line 257, in _feed
    reader_close()
    ~~~~~~~~~~~~^^
  File "/usr/lib/python3.13/multiprocessing/connection.py", line 183, in close
    self._close()
    ~~~~~~~~~~~^^
  File "/usr/lib/python3.13/multiprocessing/connection.py", line 382, in _close
    _close(self._handle)
    ~~~~~~^^^^^^^^^^^^^^
OSError: [Errno 9] Bad file descriptor

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/lib/python3.13/threading.py", line 1044, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "/usr/lib/python3.13/thr

Epoche 20 [Valid]:   0%|          | 0/3 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


 -> Resultat E20: Val Acc: 60.35%


In [ ]:
import os
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from tqdm.auto import tqdm
from PIL import Image
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd
from google.colab import drive

# --- PFADE (NUR DAS TEST-SET!) ---
CHECKPOINT_PATH = '/content/drive/MyDrive/image_only_resnet_NEW/resnet50_epoch_20.pth'
PATH_TEST_JSON  = '/content/drive/MyDrive/meme_test.json'
PATH_LABEL_MAP  = '/content/drive/MyDrive/image_only_resnet_NEW/resnet50_map.json'
PATH_IMAGES     = '/content/drive/MyDrive/all_memes/kym_memes'
SAVE_DIR        = '/content/drive/MyDrive/image_only_resnet_NEW' # Save in the new dir
OUTPUT_CSV      = 'resnet50_evaluation_results.csv'

BATCH_SIZE = 4500
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ==========================================
# 2. DATASET (NUR TEST-DATEN)
# ==========================================
class EvalResNetDataset(Dataset):
    def __init__(self, json_path, img_base_path, label_map_path):
        self.img_base_path = img_base_path
        self.samples = []

        # --- LADE DIE OFFIZIELLE LABEL MAP VOM TRAINING ---
        print(f"Lade offizielle Label-Map: {label_map_path.split('/')[-1]}")
        with open(label_map_path, 'r', encoding='utf-8') as f:
            # json speichert Keys als Strings, wir brauchen Integers für die IDs
            loaded_map = json.load(f)
            self.id_to_label = {int(k): v for k, v in loaded_map.items()}
            self.label_map = {v: int(k) for k, v in loaded_map.items()}

        # --- EVALUATION TRANSFORMS (Kein Random Crop/Flip!) ---
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

        print(f"--- [EVAL DATASET] Lade Test-Daten aus {json_path.split('/')[-1]}... ---")
        with open(json_path, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)

        for item in tqdm(raw_data, desc="Lade Bildpfade"):
            label_str = item.get('label')
            filename = item.get('filename')

            # Sicherheitscheck: Wenn ein Label in den Testdaten ist, das im Training
            # nie gesehen wurde, müssen wir es ignorieren.
            if label_str not in self.label_map:
                continue

            full_img_path = os.path.join(self.img_base_path, label_str, filename)

            if not os.path.exists(full_img_path): continue

            self.samples.append({
                'img_path': full_img_path,
                'label': self.label_map[label_str],
                'filename': filename
            })

        print(f"-> Bereit: {len(self.samples)} Test-Bilder geladen.\n")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        try:
            image = Image.open(sample['img_path']).convert("RGB")
            pixel_values = self.transform(image)
        except:
            return self.__getitem__((idx + 1) % len(self.samples))

        return {
            'pixel_values': pixel_values,
            'label': torch.tensor(sample['label'], dtype=torch.long),
            'filename': sample['filename']
        }

# ==========================================
# 3. ARCHITEKTUR (MUSS IDENTISCH ZUM TRAINING SEIN)
# ==========================================
class ResNetMemeClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        # weights=None, da wir gleich unsere eigenen Gewichte laden
        self.resnet = models.resnet50(weights=None)

        num_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(num_features, num_classes)
        )

    def forward(self, pixel_values):
        return self.resnet(pixel_values)

# ==========================================
# 4. EVALUATION LOOP
# ==========================================
def run_resnet_evaluation():
    print("--- Starte ResNet-50 Test-Evaluation ---")

    # Dataset laden
    eval_dataset = EvalResNetDataset(PATH_TEST_JSON, PATH_IMAGES, PATH_LABEL_MAP)
    eval_loader = DataLoader(eval_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    num_classes = len(eval_dataset.label_map)
    model = ResNetMemeClassifier(num_classes).to(DEVICE)

    # --- GEWICHTE LADEN ---
    if os.path.exists(CHECKPOINT_PATH):
        print(f"Lade Checkpoint: {CHECKPOINT_PATH}")
        checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)

        # Falls es ein Full-Checkpoint ist, brauchen wir nur das Modell
        if "model_state_dict" in checkpoint:
            model.load_state_dict(checkpoint["model_state_dict"])
        else:
            model.load_state_dict(checkpoint)
        print("[SUCCESS] Gewichte erfolgreich geladen.")
    else:
        print(f"[ERROR] Checkpoint nicht gefunden: {CHECKPOINT_PATH}")
        return

    model.eval()

    results_list = []
    all_preds = []
    all_labels = []

    print("Berechne Vorhersagen auf ungesehenen Daten...")

    with torch.no_grad():
        for batch in tqdm(eval_loader):
            pixel_values = batch['pixel_values'].to(DEVICE)
            labels = batch['label'].to(DEVICE)
            filenames = batch['filename']

            with torch.amp.autocast('cuda'):
                logits = model(pixel_values)

            probs = torch.softmax(logits, dim=1)
            confidences, preds = torch.max(probs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            for i in range(len(filenames)):
                pred_idx = preds[i].item()
                true_idx = labels[i].item()

                results_list.append({
                    "Dateiname": filenames[i],
                    "Wahre Klasse": eval_dataset.id_to_label[true_idx],
                    "Vorhersage": eval_dataset.id_to_label[pred_idx],
                    "Status": "KORREKT" if pred_idx == true_idx else "FALSCH",
                    "Sicherheit_Prozent": round(confidences[i].item() * 100, 2)
                })

    # Gesamte Accuracy
    acc = accuracy_score(all_labels, all_preds)
    print(f"\n========================================")
    print(f"RESNET-50 ACCURACY (Test Set): {acc:.2%}")
    print(f"========================================\n")

    # Metriken berechnen
    class_names = [eval_dataset.id_to_label[i] for i in range(num_classes)]
    report_dict = classification_report(all_labels, all_preds, target_names=class_names, output_dict=True)

    # Detail-CSV speichern
    os.makedirs(SAVE_DIR, exist_ok=True)
    df_details = pd.DataFrame(results_list)
    save_path_csv = os.path.join(SAVE_DIR, OUTPUT_CSV)
    df_details.to_csv(save_path_csv, index=False, sep=';', encoding='utf-8-sig')

    # Metriken-CSV speichern
    metrics_list = []
    for name in class_names:
        metrics = report_dict[name]
        metrics_list.append({
            "Meme": name,
            "Precision": round(metrics['precision'], 2),
            "Recall": round(metrics['recall'], 2),
            "F1-Score": round(metrics['f1-score'], 2),
            "Anzahl": metrics['support']
        })

    df_metrics = pd.DataFrame(metrics_list)
    df_metrics = df_metrics.sort_values(by="F1-Score", ascending=True)
    df_metrics.to_csv(os.path.join(SAVE_DIR, "resnet50_metrics.csv"), index=False, sep=';')

    print("[FERTIG] Tabellen gespeichert. Du kannst sie jetzt für deine Thesis auswerten!")

if __name__ == "__main__":
    run_resnet_evaluation()


--- Starte ResNet-50 Test-Evaluation ---
Lade offizielle Label-Map: resnet50_map.json
--- [EVAL DATASET] Lade Test-Daten aus meme_test.json... ---


Lade Bildpfade:   0%|          | 0/9637 [00:00<?, ?it/s]

-> Bereit: 9637 Test-Bilder geladen.

Lade Checkpoint: /content/drive/MyDrive/image_only_resnet_NEW/resnet50_epoch_20.pth
[SUCCESS] Gewichte erfolgreich geladen.
Berechne Vorhersagen auf ungesehenen Daten...


  0%|          | 0/3 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(



RESNET-50 ACCURACY (Test Set): 60.56%

[FERTIG] Tabellen gespeichert. Du kannst sie jetzt für deine Thesis auswerten!
